Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

seed = RANDOM_STATE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'opt-model'
add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 2500 # was checked with 1000 and 5000

save_model_and_metrics = True

## Optimization functions

In [5]:
def optimize_estimator_optuna(
    target:str,
    estimator:Type[BaseEstimator],
    estimator_params:dict,
    objective:callable,
    n_trials:int,
    step_scoring_average:str=step_scoring_average,
    params_to_process:list=None,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    seed:int=seed,
):
    
    # Switch off probability for SVC, since it is long to optimize
    if 'probability' in estimator_params:
        estimator_params['probability'] = False
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )
    
    opt = OptunaOptimizer(
        objective=partial(objective, ml_pipe=ml_pipe),
        study_name="model_study",
        direction="maximize",
        seed=seed,
    )
    
    opt.optimize(n_trials=n_trials)
    
    best_params = opt.study.best_params
    
    if params_to_process:
        for param in params_to_process:
            # Find one key in best_params which contains param
            key = next((k for k in best_params.keys() if param in k), None)
            if key:
                best_params[param] = best_params.pop(key)

    print('best_params')
    display(best_params)
    
    # Switch on probability for SVC, to get metrics like roc_auc for the final model
    if 'probability' in estimator_params:
        estimator_params['probability'] = True
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params={**estimator_params, **best_params},
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        step_scoring_average=step_scoring_average, # No need to pass, it would not be used
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [6]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
    'max_iter': 1000,
}
params_to_process = ['penalty']

def logreg_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    
    # # NOTE: Does not work, since optuna requires static parameters ranges
    # solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'])
    # penalty_options = {
    #     'lbfgs': ['l2'],
    #     'liblinear': ['l1', 'l2'],
    #     'newton-cg': ['l2'],
    #     'newton-cholesky': ['l2'],
    #     'sag': ['l2'],
    #     'saga': ['l1', 'l2', 'elasticnet'],
    # }
    # penalty = trial.suggest_categorical('penalty', penalty_options[solver])
    # NOTE: warnings + inconvenient to create best model
    # solver_penalty_combinations = [
    #     ("liblinear", "l1"),
    #     ("liblinear", "l2"),
    #     ("saga", "l1"),
    #     ("saga", "l2"),
    #     ("saga", "elasticnet"),
    #     ("lbfgs", "l2"),
    #     ("newton-cg", "l2"),
    #     ("newton-cholesky", "l2"),
    #     ("sag", "l2"),
    # ]
    # solver, penalty = trial.suggest_categorical(
    #     'solver_penalty', solver_penalty_combinations
    # )
    
    solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'])
    # solver = trial.suggest_categorical('solver', ['lbfgs',])
    
    if solver == 'liblinear':
        penalty = trial.suggest_categorical('penalty_liblinear', ['l1', 'l2'])
    elif solver == 'saga':
        penalty = trial.suggest_categorical('penalty_saga', ['l1', 'l2', 'elasticnet'])
    else:
        penalty = trial.suggest_categorical('penalty_other', ['l2'])
    
    # If C is large, the penalty should be small
    C = trial.suggest_float('C', 1e-4, 1e3, log=True)
    
    l1_ratio = None
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0., 1.)
    
    class_weight = trial.suggest_categorical(
        'class_weight', [None, 'balanced']
    )
    
    suggested_estimator_params = {
        "solver": solver,
        "penalty": penalty,
        "C": C,
        "l1_ratio": l1_ratio,
        "class_weight": class_weight,
    }
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=logreg_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 11:26:05,329] A new study created in memory with name: model_study
[I 2025-04-15 11:26:05,482] Trial 0 finished with value: 0.8080335848738224 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l2', 'C': 1.6136341713591311, 'class_weight': None}. Best is trial 0 with value: 0.8080335848738224.
[I 2025-04-15 11:26:05,640] Trial 1 finished with value: 0.8161809420899051 and parameters: {'solver': 'lbfgs', 'penalty_other': 'l2', 'C': 0.47129737561107793, 'class_weight': None}. Best is trial 1 with value: 0.8161809420899051.
[I 2025-04-15 11:26:05,785] Trial 2 finished with value: 0.26121337827326935 and parameters: {'solver': 'saga', 'penalty_saga': 'elasticnet', 'C': 0.00021142332035497166, 'l1_ratio': 0.6075448519014384, 'class_weight': None}. Best is trial 1 with value: 0.8161809420899051.
[I 2025-04-15 11:26:05,934] Trial 3 finished with value: 0.8030685635144312 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l1', 'C': 0.292575779498244, 'class_

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_opt-model
holdout_test_f1_macro,0.759479
holdout_test_accuracy_balanced,0.766204
holdout_test_roc_auc,0.886574
holdout_test_f1,0.817204
holdout_test_accuracy,0.773333
cv_test_f1_macro_median,0.799242
cv_test_accuracy_balanced_median,0.816667
cv_test_roc_auc_median,0.868421


Model saved in ..\results\models_modelling4\LogisticRegression_splashing_smote_opt-model


In [7]:
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
    'max_iter': 1000,
}

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=logreg_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 11:33:12,355] A new study created in memory with name: model_study
[I 2025-04-15 11:33:12,481] Trial 0 finished with value: 0.802106586533207 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l2', 'C': 1.6136341713591311, 'class_weight': None}. Best is trial 0 with value: 0.802106586533207.
[I 2025-04-15 11:33:12,619] Trial 1 finished with value: 0.7922274233971268 and parameters: {'solver': 'lbfgs', 'penalty_other': 'l2', 'C': 0.47129737561107793, 'class_weight': None}. Best is trial 0 with value: 0.802106586533207.
[I 2025-04-15 11:33:12,747] Trial 2 finished with value: 0.4233049487844008 and parameters: {'solver': 'saga', 'penalty_saga': 'elasticnet', 'C': 0.00021142332035497166, 'l1_ratio': 0.6075448519014384, 'class_weight': None}. Best is trial 0 with value: 0.802106586533207.
[I 2025-04-15 11:33:12,873] Trial 3 finished with value: 0.7953914760068441 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l1', 'C': 0.292575779498244, 'class_weigh

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_smote_opt-...
holdout_test_f1_macro,0.802991
holdout_test_accuracy_balanced,0.85
holdout_test_roc_auc,0.950909
holdout_test_f1,0.734694
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.826797
cv_test_accuracy_balanced_median,0.864469
cv_test_roc_auc_median,0.943223


Model saved in ..\results\models_modelling4\LogisticRegression_no_fragmentation_smote_opt-model


# KNClassifier

In [8]:
estimator = KNeighborsClassifier
target = 'splashing'
estimator_params = {}
# params_to_process = ['penalty']
params_to_process = None


def knn_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    n_neighbors = trial.suggest_int("n_neighbors", 1, 50)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    p = trial.suggest_int("p", 1, 2)  # 1=manhattan, 2=euclidean
    # leaf_size - affects the speed of the tree construction and search
    # algorithm - affects the speed of the search, for small datasets it is okay to use auto
    
    suggested_estimator_params = {
        "n_neighbors": n_neighbors,
        "weights": weights,
        "p": p,
    }
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=knn_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 11:39:33,575] A new study created in memory with name: model_study
[I 2025-04-15 11:39:33,832] Trial 0 finished with value: 0.7977895578771674 and parameters: {'n_neighbors': 19, 'weights': 'uniform', 'p': 2}. Best is trial 0 with value: 0.7977895578771674.
[I 2025-04-15 11:39:34,062] Trial 1 finished with value: 0.8009871998340693 and parameters: {'n_neighbors': 8, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.8009871998340693.
[I 2025-04-15 11:39:34,298] Trial 2 finished with value: 0.7824376182535576 and parameters: {'n_neighbors': 31, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.8009871998340693.
[I 2025-04-15 11:39:34,532] Trial 3 finished with value: 0.7635110652728538 and parameters: {'n_neighbors': 42, 'weights': 'uniform', 'p': 1}. Best is trial 1 with value: 0.8009871998340693.
[I 2025-04-15 11:39:34,792] Trial 4 finished with value: 0.7825674250841211 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'p': 1}. Best is trial

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_smote_opt-model
holdout_test_f1_macro,0.8333
holdout_test_accuracy_balanced,0.820602
holdout_test_roc_auc,0.880015
holdout_test_f1,0.891089
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.896201
cv_test_accuracy_balanced_median,0.891641
cv_test_roc_auc_median,0.916541


Model saved in ..\results\models_modelling4\KNeighborsClassifier_splashing_smote_opt-model


In [9]:
estimator = KNeighborsClassifier
target = 'no_fragmentation'
estimator_params = {}
params_to_process = None

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=knn_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 11:47:25,188] A new study created in memory with name: model_study
[I 2025-04-15 11:47:25,402] Trial 0 finished with value: 0.7495331386902663 and parameters: {'n_neighbors': 19, 'weights': 'uniform', 'p': 2}. Best is trial 0 with value: 0.7495331386902663.
[I 2025-04-15 11:47:25,624] Trial 1 finished with value: 0.7939749558884351 and parameters: {'n_neighbors': 8, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.7939749558884351.
[I 2025-04-15 11:47:25,831] Trial 2 finished with value: 0.7300499179946813 and parameters: {'n_neighbors': 31, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.7939749558884351.
[I 2025-04-15 11:47:26,052] Trial 3 finished with value: 0.7201734561003093 and parameters: {'n_neighbors': 42, 'weights': 'uniform', 'p': 1}. Best is trial 1 with value: 0.7939749558884351.
[I 2025-04-15 11:47:26,276] Trial 4 finished with value: 0.7674836215380793 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'p': 1}. Best is trial

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_smote_op...
holdout_test_f1_macro,0.811873
holdout_test_accuracy_balanced,0.843182
holdout_test_roc_auc,0.903182
holdout_test_f1,0.73913
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.865709
cv_test_accuracy_balanced_median,0.887363
cv_test_roc_auc_median,0.92674


Model saved in ..\results\models_modelling4\KNeighborsClassifier_no_fragmentation_smote_opt-model


# SVC

In [10]:
estimator = SVC
target = 'splashing'
estimator_params = {
    'probability': True, # Would beFalse for optimization, True for metrics
    # 'shrinking': True, # already in the default params
}
params_to_process = ['gamma']


def svc_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    kernel = trial.suggest_categorical(
        'kernel', ['linear', 'rbf', 'sigmoid', 'poly']
    )
    C = trial.suggest_float('C', 1e-3, 1e3, log=True)
    
    class_weight = trial.suggest_categorical(
        'class_weight', [None, 'balanced']
    )
    
    suggested_estimator_params = {
        "kernel": kernel,
        "C": C,
        "class_weight": class_weight,
        # Optional params
        # 'gamma': gamma
        # 'coef0': coef0,
        # 'degree': degree,
    }
    
    if kernel in ['rbf', 'poly', 'sigmoid']:
        gamma_choice = trial.suggest_categorical(
            "gamma_type", ["scale", "auto", "float"]
        )
        if gamma_choice == "float":
            gamma = trial.suggest_float("gamma", 1e-4, 10, log=True)
        else:
            gamma = gamma_choice
        suggested_estimator_params['gamma'] = gamma
    
    # Optional params
    if kernel in ['sigmoid', 'poly']:
        suggested_estimator_params['coef0'] = trial.suggest_float(
            "coef0", 0.0, 1.0
        )
    if kernel == 'poly':
        suggested_estimator_params['degree'] = trial.suggest_int(
            "degree", 2, 5
        )
    
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    # print(estimator_params)
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=svc_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 11:56:50,131] A new study created in memory with name: model_study
[I 2025-04-15 11:56:50,362] Trial 0 finished with value: 0.8080241913509035 and parameters: {'kernel': 'rbf', 'C': 0.008632008168602538, 'class_weight': None, 'gamma_type': 'scale'}. Best is trial 0 with value: 0.8080241913509035.
[I 2025-04-15 11:56:50,592] Trial 1 finished with value: 0.8080241913509035 and parameters: {'kernel': 'rbf', 'C': 0.012329623163659839, 'class_weight': 'balanced', 'gamma_type': 'scale'}. Best is trial 0 with value: 0.8080241913509035.
[I 2025-04-15 11:56:50,744] Trial 2 finished with value: 0.8296865496001313 and parameters: {'kernel': 'linear', 'C': 0.5450293694558254, 'class_weight': None}. Best is trial 2 with value: 0.8296865496001313.
[I 2025-04-15 11:56:50,918] Trial 3 finished with value: 0.42602998654463675 and parameters: {'kernel': 'poly', 'C': 0.010547383621352036, 'class_weight': 'balanced', 'gamma_type': 'scale', 'coef0': 0.09767211400638387, 'degree': 4}. Best is 

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "SVC"


,0
target,splashing
model,SVC_splashing_smote_opt-model
holdout_test_f1_macro,0.806892
holdout_test_accuracy_balanced,0.799769
holdout_test_roc_auc,0.892747
holdout_test_f1,0.868687
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.898584
cv_test_accuracy_balanced_median,0.903251
cv_test_roc_auc_median,0.927245


Model saved in ..\results\models_modelling4\SVC_splashing_smote_opt-model


In [6]:
estimator = SVC
target = 'no_fragmentation'
estimator_params = {
    'probability': True, # Would be False for optimization, True for metrics
    # 'shrinking': True, # already in the default params
}
params_to_process = ['gamma']

def svc_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    kernel = trial.suggest_categorical(
        'kernel', ['linear', 'rbf', 'sigmoid', 'poly']
    )
    C = trial.suggest_float('C', 1e-3, 1e3, log=True)
    
    class_weight = trial.suggest_categorical(
        'class_weight', [None, 'balanced']
    )
    
    suggested_estimator_params = {
        "kernel": kernel,
        "C": C,
        "class_weight": class_weight,
        # Optional params
        # 'gamma': gamma
        # 'coef0': coef0,
        # 'degree': degree,
    }
    
    if kernel in ['rbf', 'poly', 'sigmoid']:
        gamma_choice = trial.suggest_categorical(
            "gamma_type", ["scale", "auto", "float"]
        )
        if gamma_choice == "float":
            gamma = trial.suggest_float("gamma", 1e-4, 10, log=True)
        else:
            gamma = gamma_choice
        suggested_estimator_params['gamma'] = gamma
    
    # Optional params
    if kernel in ['sigmoid', 'poly']:
        suggested_estimator_params['coef0'] = trial.suggest_float(
            "coef0", 0.0, 1.0
        )
    if kernel == 'poly':
        suggested_estimator_params['degree'] = trial.suggest_int(
            "degree", 2, 5
        )
    
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    # print(estimator_params)
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=svc_objective,
    n_trials=1400,
    seed=seed,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 12:41:50,248] A new study created in memory with name: model_study
[I 2025-04-15 12:41:50,480] Trial 0 finished with value: 0.7887325010206364 and parameters: {'kernel': 'rbf', 'C': 0.008632008168602538, 'class_weight': None, 'gamma_type': 'scale'}. Best is trial 0 with value: 0.7887325010206364.
[I 2025-04-15 12:41:50,699] Trial 1 finished with value: 0.7887325010206364 and parameters: {'kernel': 'rbf', 'C': 0.012329623163659839, 'class_weight': 'balanced', 'gamma_type': 'scale'}. Best is trial 0 with value: 0.7887325010206364.
[I 2025-04-15 12:41:50,859] Trial 2 finished with value: 0.827403676065338 and parameters: {'kernel': 'linear', 'C': 0.5450293694558254, 'class_weight': None}. Best is trial 2 with value: 0.827403676065338.
[I 2025-04-15 12:41:51,031] Trial 3 finished with value: 0.42566619835948 and parameters: {'kernel': 'poly', 'C': 0.010547383621352036, 'class_weight': 'balanced', 'gamma_type': 'scale', 'coef0': 0.09767211400638387, 'degree': 4}. Best is trial

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "SVC"


,0
target,no_fragmentation
model,SVC_no_fragmentation_smote_opt-model
holdout_test_f1_macro,0.839194
holdout_test_accuracy_balanced,0.861364
holdout_test_roc_auc,0.949091
holdout_test_f1,0.772727
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.871224
cv_test_accuracy_balanced_median,0.913004
cv_test_roc_auc_median,0.974359


Model saved in ..\results\models_modelling4\SVC_no_fragmentation_smote_opt-model
